In [1]:
# T4 GPU(무료) 환경에서 4-bit 양자화 및 LangChain/LangGraph 호환을 위함
!pip install "langchain>=0.2.1" "langgraph>=0.0.60" "langchain-huggingface>=0.0.3" \
"langchain_community>=0.2.1" "transformers>=4.41.2" "bitsandbytes>=0.43.1" \
"accelerate>=0.30.1" "pandas>=2.0.0" "openpyxl>=3.1.2" "torch>=2.3.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
# -*- coding: utf-8 -*-
"""
LangChain + LangGraph 기반 배치 추론 + RAG + CSV 스트리밍 + 안전 재시작 (오류 방지 강화판)
+ (선택) 분류 모델 학습(train) + EDA + loss/accuracy/ROC/NDCG + 체크포인트 재시작

요약:
- 먼저 엑셀 로드 & EDA(NA 비율, 텍스트 길이 분포, 라벨 분포)
- (라벨 컬럼이 있으면) 분류 모델을 학습하고 loss/accuracy/f1/ROC/NDCG 지표 계산
    - HuggingFace Trainer 사용 → 체크포인트 자동 저장
    - Colab 끊겼다가 다시 실행해도 마지막 checkpoint에서 이어서 학습
- 그 다음, 기존처럼 LLM(Gemma/Qwen)으로 요약을 배치 생성
    - CSV 스트리밍 + row_id 기반 재시작
"""

from __future__ import annotations
import os, sys, time, csv, signal
from pathlib import Path
from typing import List, Dict, Any, Set, TypedDict

import numpy as np
import pandas as pd
import torch

# EDA / 학습용
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import ndcg_score

# ─────────────────────────────────────────────────────────
# 0) Google Drive 마운트
# ─────────────────────────────────────────────────────────
IN_COLAB = "COLAB_GPU" in os.environ or (Path("/content").exists())
if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive").exists():
            drive.mount('/content/drive')
            print("[GDRIVE] Mounted at /content/drive")
        else:
            print("[GDRIVE] Drive already mounted.")
    except Exception as e:
        print(f"[GDRIVE][WARN] Drive mount 실패(무시 가능): {e}")
else:
    print("[INFO] Not in Colab. GDrive Mount를 건너뜁니다.")

# ─────────────────────────────────────────────────────────
# 1) 경로 설정
# ─────────────────────────────────────────────────────────
# (필요 시 아래 경로들만 본인 환경에 맞게 수정)

# 요약용 입력 엑셀
INPUT_FILE_PATH  = "/content/drive/MyDrive/여성맞춤정책_요약_2차_결과_병합.xlsx"

# 요약 결과 저장
OUTPUT_CSV_PATH  = "/content/drive/MyDrive/policy_summary_langchain_streaming.csv"
OUTPUT_XLSX_PATH = "/content/drive/MyDrive/policy_summary_langchain_final.xlsx"

# (선택) 분류 모델 학습 결과 저장
TRAIN_ROOT        = Path("/content/drive/MyDrive/policy_train_runs")
TRAIN_CKPT_DIR    = TRAIN_ROOT / "checkpoints"
TRAIN_LOG_DIR     = TRAIN_ROOT / "logs"
TRAIN_METRIC_CSV  = TRAIN_ROOT / "metrics.csv"

INPUT_PATH = Path(INPUT_FILE_PATH)

# 요약에 필요한 필수 컬럼
REQUIRED_COLS = [
    "지원대상_원문", "지원내용_원문",
    "지원대상_초벌요약", "지원내용_초벌요약",
]

# (선택) 학습에 사용할 텍스트/라벨 컬럼 이름
# ▶ 기본값은 예시임. 실제 라벨 컬럼명이 있으면 여기를 수정하면 됨.
TEXT_COL_TRAIN  = "지원내용_원문"
LABEL_COL_TRAIN = "label"  # 실제 파일에 없으면 학습 파트는 자동으로 스킵됨

ENABLE_TRAINING   = True   # 학습기능 활성화 여부
RESUME_TRAINING   = True   # 체크포인트에서 재시작
TRAIN_NUM_EPOCHS  = 3
TRAIN_BATCH_SIZE  = 8
TRAIN_LR          = 5e-5
CLS_MODEL_NAME    = os.environ.get("CLS_MODEL_NAME", "klue/roberta-base").strip()

# ─────────────────────────────────────────────────────────
# 2) 입력 파일/컬럼 Fail-Fast + EDA
# ─────────────────────────────────────────────────────────
if not INPUT_PATH.is_file():
    print(f"[ERR] 파일을 찾을 수 없습니다: {INPUT_PATH}", file=sys.stderr)
    sys.exit(1)

try:
    df = pd.read_excel(INPUT_PATH)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        print(f"[ERR] 필수 컬럼 누락: {missing}", file=sys.stderr)
        print(f"[INFO] 현재 컬럼: {df.columns.tolist()}", file=sys.stderr)
        sys.exit(1)
    print(f"[XLSX] Loaded: {INPUT_PATH} (rows={len(df)})")
except Exception as e:
    print(f"[ERR] 엑셀 파일 로드 실패: {e}", file=sys.stderr)
    raise

# ---- EDA 1: NA 비율 ----
print("\n[EDA] NA 비율 (주요 텍스트 컬럼):")
print(df[["지원대상_원문", "지원내용_원문",
          "지원대상_초벌요약", "지원내용_초벌요약"]].isna().mean())

# ---- EDA 2: 텍스트 길이 분포 ----
print("\n[EDA] 텍스트 길이 분포:")
for col in ["지원대상_원문", "지원내용_원문"]:
    lens = df[col].astype(str).str.len()
    print(f"\n  ▷ {col} 길이 통계:")
    print(lens.describe())

# ---- EDA 3: 라벨 분포 (라벨 컬럼이 있을 때만) ----
if LABEL_COL_TRAIN in df.columns:
    print(f"\n[EDA] 라벨 분포 (LABEL_COL_TRAIN={LABEL_COL_TRAIN}):")
    print(df[LABEL_COL_TRAIN].value_counts())
else:
    print(f"\n[EDA] 라벨 컬럼 '{LABEL_COL_TRAIN}' 이(가) 없어서 라벨 분포/학습 파트는 스킵됩니다.")

# ─────────────────────────────────────────────────────────
# 2-1) (선택) 분류 모델 학습 + 지표 + 체크포인트 재시작
# ─────────────────────────────────────────────────────────
if ENABLE_TRAINING and LABEL_COL_TRAIN in df.columns:
    print("\n[TRAIN] 분류 모델 학습을 시작합니다...")

    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        DataCollatorWithPadding,
        TrainingArguments,
        Trainer,
    )
    from transformers.trainer_utils import get_last_checkpoint, set_seed

    set_seed(42)

    # NA 제거
    train_df_full = df.dropna(subset=[TEXT_COL_TRAIN, LABEL_COL_TRAIN]).reset_index(drop=True)
    print(f"[TRAIN] NA 제거 후 학습용 데이터 shape = {train_df_full.shape}")

    # 라벨 인코딩
    le = LabelEncoder()
    train_df_full["label_id"] = le.fit_transform(train_df_full[LABEL_COL_TRAIN].astype(str))
    num_labels = len(le.classes_)
    print("[TRAIN] 라벨 인코딩:", dict(zip(range(num_labels), le.classes_)))

    # train/val/test 분할
    try:
        tr_df, tmp_df = train_test_split(
            train_df_full,
            test_size=0.2,
            stratify=train_df_full["label_id"],
            random_state=42,
        )
        val_df, te_df = train_test_split(
            tmp_df,
            test_size=0.5,
            stratify=tmp_df["label_id"],
            random_state=42,
        )
    except ValueError as e:
        print(f"[TRAIN][WARN] stratify 실패 → 일반 분할 사용: {e}")
        tr_df, tmp_df = train_test_split(
            train_df_full,
            test_size=0.2,
            random_state=42,
        )
        val_df, te_df = train_test_split(
            tmp_df,
            test_size=0.5,
            random_state=42,
        )

    print(f"[TRAIN] split: train={len(tr_df)}, val={len(val_df)}, test={len(te_df)}")

    # 토크나이저/데이터셋
    cls_tokenizer = AutoTokenizer.from_pretrained(CLS_MODEL_NAME)

    class TextClsDataset(torch.utils.data.Dataset):
        def __init__(self, df: pd.DataFrame, text_col: str, label_col: str):
            self.df = df.reset_index(drop=True)
            self.text_col = text_col
            self.label_col = label_col

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx: int) -> Dict[str, Any]:
            row = self.df.iloc[idx]
            text = str(row[self.text_col])
            label = int(row[self.label_col])

            enc = cls_tokenizer(
                text,
                truncation=True,
                max_length=256,
                padding=False,
            )
            enc["labels"] = label
            return enc

    train_ds = TextClsDataset(tr_df, TEXT_COL_TRAIN, "label_id")
    val_ds   = TextClsDataset(val_df, TEXT_COL_TRAIN, "label_id")
    test_ds  = TextClsDataset(te_df, TEXT_COL_TRAIN, "label_id")

    data_collator = DataCollatorWithPadding(tokenizer=cls_tokenizer)

    # 모델
    cls_model = AutoModelForSequenceClassification.from_pretrained(
        CLS_MODEL_NAME,
        num_labels=num_labels,
    )
    cls_model.to("cuda" if torch.cuda.is_available() else "cpu")

    # 지표 함수
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        acc = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average="macro")

        # softmax 확률
        probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

        # ROC-AUC (multi-class, macro OVR)
        try:
            y_true_onehot = np.eye(num_labels)[labels]
            roc_auc = roc_auc_score(y_true_onehot, probs, multi_class="ovr", average="macro")
        except Exception as e:
            print(f"[METRIC][WARN] ROC-AUC 계산 실패: {e}")
            roc_auc = float("nan")

        # NDCG
        try:
            y_true_rel = np.eye(num_labels)[labels]
            ndcg = ndcg_score(y_true_rel, probs)
        except Exception as e:
            print(f"[METRIC][WARN] NDCG 계산 실패: {e}")
            ndcg = float("nan")

        return {
            "accuracy": acc,
            "f1_macro": f1,
            "roc_auc_macro_ovr": roc_auc,
            "ndcg": ndcg,
        }

    # TrainingArguments + 체크포인트
    TRAIN_ROOT.mkdir(parents=True, exist_ok=True)
    TRAIN_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    TRAIN_LOG_DIR.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(TRAIN_CKPT_DIR),
        num_train_epochs=TRAIN_NUM_EPOCHS,
        learning_rate=TRAIN_LR,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=max(TRAIN_BATCH_SIZE, 16),
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_dir=str(TRAIN_LOG_DIR),
        fp16=torch.cuda.is_available(),
        report_to=["none"],
    )

    trainer = Trainer(
        model=cls_model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=cls_tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    last_ckpt = None
    if TRAIN_CKPT_DIR.is_dir():
        last_ckpt = get_last_checkpoint(str(TRAIN_CKPT_DIR))
        if last_ckpt:
            print(f"[TRAIN][RESUME] 마지막 체크포인트 발견: {last_ckpt}")

    # 학습 실행 (중단되면 다음에 다시 실행 시 last_ckpt에서 이어서)
    if RESUME_TRAINING and last_ckpt:
        print("[TRAIN] 체크포인트에서 이어서 학습합니다...")
        train_result = trainer.train(resume_from_checkpoint=last_ckpt)
    else:
        print("[TRAIN] 처음부터 학습합니다...")
        train_result = trainer.train()

    trainer.save_model()
    train_metrics = train_result.metrics
    print("\n[TRAIN] final train metrics:", train_metrics)

    print("\n[EVAL] Validation set:")
    eval_metrics = trainer.evaluate(eval_dataset=val_ds)
    print(eval_metrics)

    print("\n[TEST] Test set:")
    test_metrics = trainer.evaluate(eval_dataset=test_ds)
    print(test_metrics)

    # 메트릭 CSV 저장
    metrics_df = pd.DataFrame([
        {"split": "train", **train_metrics},
        {"split": "val",   **eval_metrics},
        {"split": "test",  **test_metrics},
    ])
    TRAIN_METRIC_CSV.parent.mkdir(parents=True, exist_ok=True)
    metrics_df.to_csv(TRAIN_METRIC_CSV, index=False, encoding="utf-8-sig")
    print(f"\n[TRAIN] 메트릭 CSV 저장: {TRAIN_METRIC_CSV}")

else:
    print("\n[TRAIN] 학습 기능 비활성화이거나, 라벨 컬럼이 없어 학습을 건너뜁니다.")

# ─────────────────────────────────────────────────────────
# 3) LangChain / LangGraph (요약 파이프라인)
#    - LangGraph 없으면 fallback으로 진행
# ─────────────────────────────────────────────────────────
use_langgraph = False

try:
    import langchain
    from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
    from langchain_core.output_parsers import StrOutputParser

    try:
        import langgraph
        from langgraph.graph import StateGraph
        use_langgraph = True
        print(f"\nLangChain: {langchain.__version__}, LangGraph: {langgraph.__version__} import 성공")
    except Exception as e:
        print(f"[WARN] LangGraph import 실패 → LangGraph 없이 진행합니다: {e}")
        use_langgraph = False

except Exception as e:
    print(f"[ERR] LangChain import 실패: {e}", file=sys.stderr)
    sys.exit(1)

# ─────────────────────────────────────────────────────────
# 4) LLM 로딩 (요약용 LLM, 4bit)
# ─────────────────────────────────────────────────────────
MODEL_ID = os.environ.get("MODEL_ID", "google/gemma-2-9b-it").strip()
FALLBACK_MODEL_ID = os.environ.get("FALLBACK_MODEL_ID", "Qwen/Qwen2.5-7B-Instruct").strip()
HF_TOKEN = (os.environ.get("HF_TOKEN", "").strip())

BATCH_SIZE     = int(os.environ.get("BATCH_SIZE", "4"))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "300"))
DO_SAMPLE      = os.environ.get("DO_SAMPLE", "false").lower() == "true"

print("\n====== RUNTIME CHECK (요약용) ======")
print("Python:", sys.version)
print("CUDA avail? ", torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        print("GPU:", torch.cuda.get_device_name(0))
    except Exception:
        pass
print("====================================\n")

try:
    import transformers
    from transformers import AutoTokenizer, AutoModelForCausalLM
    print("transformers :", transformers.__version__)
except Exception as e:
    print(f"[ERR] transformers import 실패: {e}")
    raise

_has_bnb = False
try:
    import bitsandbytes as bnb  # noqa: F401
    from transformers import BitsAndBytesConfig
    _has_bnb = True
    print("bitsandbytes : installed (4-bit 양자화 가능)")
except Exception as e:
    print(f"bitsandbytes : not installed ({e})")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def may_need_login(model_id: str) -> bool:
    return any(k in model_id.lower() for k in ["gemma", "google/"])

def hf_login_if_needed():
    if not HF_TOKEN and may_need_login(MODEL_ID):
        print("[HF] HF_TOKEN 없음. 게이트드 모델 접근 실패 시 폴백합니다.")
        return
    if HF_TOKEN:
        try:
            from huggingface_hub import login
            login(token=HF_TOKEN, add_to_git_credential=False)
            print("[HF] 로그인 성공(HF_TOKEN).")
        except Exception as e:
            print(f"[HF][WARN] 로그인 실패(계속 진행): {e}")

hf_login_if_needed()

def postprocess_summary(text: str) -> str:
    """LLM 출력이 쉼표나 중간에서 끊긴 경우를 최소화하기 위한 후처리."""
    s = (text or "").strip()
    if not s:
        return s

    # 혹시 프롬프트 일부가 섞여 나왔으면 잘라내기 (방어용)
    cut_markers = ["[지시]", "[원본 텍스트", "[참고용 초벌 요약"]
    for m in cut_markers:
        idx = s.find(m)
        if idx > 0:
            s = s[:idx].strip()

    # 끝에 이상한 기호만 있는 경우 정리
    while s and s[-1] in ",·;:/-":
        s = s[:-1].rstrip()

    # 1) 한국어 문장 종결 패턴 기준으로 마지막 완전한 문장까지만 남기기
    endings = ["다.", "니다.", "합니다.", "해요.", "됩니다.", "돼요.", "임.", "함."]
    for e in endings:
        idx = s.rfind(e)
        if idx != -1:
            return s[:idx + len(e)].strip()

    # 2) 그래도 없으면 일반 마침표/느낌표/물음표 기준
    for e in [".", "!", "?"]:
        idx = s.rfind(e)
        if idx != -1:
            return s[:idx + 1].strip()

    # 3) 끝이 쉼표/접속사류로 끝나면 그냥 마침표로 강제 마무리
    if s[-1] in {",", "와", "및"}:
        s = s.rstrip(", 와및").rstrip()

    return s + "다."

def load_model_and_tokenizer(model_id: str):
    print(f"[LOAD] Loading model: {model_id}")
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True, token=HF_TOKEN or None)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    if _has_bnb and torch.cuda.is_available():
        try:
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16
            )
            mdl = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_cfg,
                device_map="auto",
                torch_dtype=torch.bfloat16,
                token=HF_TOKEN or None
            )
            print("[QNT] 4-bit quantization enabled.")
            return tok, mdl
        except Exception as e:
            print(f"[QNT][WARN] 4-bit 실패 → FP/BF16로 폴백: {e}")

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto" if torch.cuda.is_available() else None,
        torch_dtype=dtype,
        token=HF_TOKEN or None
    )
    if not torch.cuda.is_available():
        mdl.to(DEVICE)
    return tok, mdl

try:
    tokenizer, model = load_model_and_tokenizer(MODEL_ID)
    print("[LOAD] Primary model loaded.\n")
except Exception as e:
    print(f"[LOAD][ERR] Primary 실패: {e}", file=sys.stderr)
    print(f"[LOAD] Fallback 시도: {FALLBACK_MODEL_ID}")
    tokenizer, model = load_model_and_tokenizer(FALLBACK_MODEL_ID)
    print("[LOAD] Fallback model loaded.\n")

model.eval()

# ─────────────────────────────────────────────────────────
# 5) 프롬프트(챗 템플릿) + LangGraph 설정 (fallback 포함)
# ─────────────────────────────────────────────────────────
COMBINED_PROMPT_TEXT = """
[지시]
당신은 대한민국 정부의 정책 문서를 요약하는 전문가입니다.
주어진 '원본 텍스트'(지원대상_원문, 지원내용_원문)의 핵심을 빠뜨리지 말고,
'참고용 초벌 요약'을 스타일 참조로 삼되 부족한 부분은 보완하여,
2~3문장 내의 자연스럽고 문법적으로 정확한 '최종 요약문'을 작성하세요.
항상 완전한 문장으로 작성하고, 마지막 문장은 반드시 마침표(예: "…제도입니다.", "…을 지원합니다.")로 끝내세요.
수동태·번역투를 피하고, 대상과 내용을 함께 명확히 서술하세요.

[원본 텍스트 (RAG Context)]
- 지원대상_원문: {target_text}
- 지원내용_원문: {content_text}

[참고용 초벌 요약]
- 지원대상_초벌요약: {target_draft}
- 지원내용_초벌요약: {content_draft}

[최종 요약]
""".strip()

prompt_template = ChatPromptTemplate.from_messages([
    HumanMessagePromptTemplate.from_template(COMBINED_PROMPT_TEXT)
])
_ = StrOutputParser()

class GraphState(TypedDict):
    target_text: str
    content_text: str
    target_draft: str
    content_draft: str
    summary: str

def _safe_str(x) -> str:
    if pd.isna(x) or x is None:
        return ""
    return str(x)

def generate_summary_node(state: GraphState) -> Dict[str, str]:
    try:
        prompt_text = COMBINED_PROMPT_TEXT.format(
            target_text=_safe_str(state.get("target_text", "")),
            content_text=_safe_str(state.get("content_text", "")),
            target_draft=_safe_str(state.get("target_draft", "")),
            content_draft=_safe_str(state.get("content_draft", "")),
        )
        messages = [{"role": "user", "content": prompt_text}]

        input_ids = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(DEVICE)

        gen_kwargs = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.eos_token_id
        )
        if DO_SAMPLE:
            gen_kwargs.update(dict(temperature=0.3, top_p=0.9))

        outputs = model.generate(
            input_ids,
            eos_token_id=tokenizer.eos_token_id,  # ★ EOS 지정
            **gen_kwargs,
        )
        response = outputs[0][input_ids.shape[-1]:]
        summary_raw = tokenizer.decode(response, skip_special_tokens=True).strip()
        summary = postprocess_summary(summary_raw)
        if not summary:
            summary = "정보 없음(오류)"
        return {"summary": summary}
    except Exception as e:
        print(f"[LG][ERR] 노드 실행 실패: {repr(e)}", file=sys.stderr)
        return {"summary": "정보 없음(오류)"}

# LangGraph 사용 여부에 따라 분기
if use_langgraph:
    workflow = StateGraph(GraphState)  # type: ignore[name-defined]
    workflow.add_node("generate", generate_summary_node)
    workflow.set_entry_point("generate")
    workflow.set_finish_point("generate")
    app = workflow.compile()
    print("[LG] LangGraph 워크플로우 컴파일 완료.")

    def generate_batch(batch_inputs: List[Dict[str, Any]]) -> List[str]:
        graph_inputs = [{**b_in, "summary": ""} for b_in in batch_inputs]
        try:
            results = app.batch(graph_inputs, config={"max_concurrency": BATCH_SIZE})
            return [res.get("summary", "정보 없음(오류)") for res in results]
        except Exception as e:
            print(f"[LG][FATAL] LangGraph 배치 생성 실패: {e}", file=sys.stderr)
            return ["정보 없음(배치오류)"] * len(batch_inputs)

else:
    print("[LG][WARN] LangGraph 없이 순차/배치 for-loop로 요약을 생성합니다.")

    def generate_batch(batch_inputs: List[Dict[str, Any]]) -> List[str]:
        summaries: List[str] = []
        for b_in in batch_inputs:
            state: GraphState = {
                "target_text": _safe_str(b_in.get("target_text", "")),
                "content_text": _safe_str(b_in.get("content_text", "")),
                "target_draft": _safe_str(b_in.get("target_draft", "")),
                "content_draft": _safe_str(b_in.get("content_draft", "")),
                "summary": "",
            }
            out = generate_summary_node(state)
            summaries.append(out.get("summary", "정보 없음(오류)"))
        return summaries

# ─────────────────────────────────────────────────────────
# 7) CSV 스트리밍 + resume (오류행 자동 재시도)
# ─────────────────────────────────────────────────────────
OUT_CSV_PATH = Path(OUTPUT_CSV_PATH)
ROW_ID_COL = "row_id"
OUT_COL = "최종요약_final"
out_columns = [ROW_ID_COL] + list(df.columns) + [OUT_COL]

def ensure_csv_header(path: Path, header: List[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists() or path.stat().st_size == 0:
        with path.open("w", newline="", encoding="utf-8-sig") as f:
            csv.writer(f).writerow(header)
        print(f"[OUT] CSV 헤더 생성: {path}")

def _is_bad(val: str) -> bool:
    if val is None:
        return True
    s = str(val).strip()
    return (not s) or s in {"정보 없음(오류)", "정보 없음(배치오류)", "정보 없음"}

def read_processed_ids_strict(path: Path, out_col: str) -> Set[int]:
    done: Set[int] = set()
    if not path.exists() or path.stat().st_size == 0:
        return done
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        rdr = csv.reader(f)
        header = next(rdr, None)
        if not header:
            return done
        try:
            ridx = header.index(ROW_ID_COL)
            oidx = header.index(out_col)
        except ValueError:
            return set()
        for row in rdr:
            if not row or len(row) <= max(ridx, oidx):
                continue
            try:
                rid = int(row[ridx])
            except Exception:
                continue
            outv = row[oidx]
            if not _is_bad(outv):
                done.add(rid)
    return done

FORCE_REDO_ALL = os.environ.get("FORCE_REDO_ALL", "false").lower() == "true"

ensure_csv_header(OUT_CSV_PATH, out_columns)
processed_ids = read_processed_ids_strict(OUT_CSV_PATH, OUT_COL)
if FORCE_REDO_ALL:
    processed_ids = set()
    print("[RESUME] FORCE_REDO_ALL=True → 모든 행 재처리")
print(f"[RESUME] (정상요약 기준) 처리 완료된 row_id 수: {len(processed_ids)}")

# 안전 종료 시그널
_stop = False
def _sigterm_handler(signum, frame):
    global _stop
    _stop = True
    print("\n[SIG] 안전 종료 요청 수신. 현재 배치 처리 후 중단합니다.")

signal.signal(signal.SIGINT, _sigterm_handler)
signal.signal(signal.SIGTERM, _sigterm_handler)

# ─────────────────────────────────────────────────────────
# 8) 메인 루프: 배치 추론 + 행당 CSV 즉시 저장
# ─────────────────────────────────────────────────────────
t0 = time.time()
total = len(df)
print(f"[RUN] 요약 시작 (model={MODEL_ID} | batch={BATCH_SIZE})")

with OUT_CSV_PATH.open("a", newline="", encoding="utf-8-sig") as f_csv:
    writer = csv.writer(f_csv)

    next_start = 0
    processed = len(processed_ids)

    while next_start < total and not _stop:
        batch_row_ids: List[int] = []
        batch_inputs: List[Dict[str, Any]] = []
        rows_cache: List[List[Any]] = []

        i = next_start
        while i < total and len(batch_row_ids) < BATCH_SIZE:
            if i in processed_ids:
                i += 1
                continue

            row = df.iloc[i]
            input_data = {
                "target_text": _safe_str(row["지원대상_원문"]),

                "content_text": _safe_str(row["지원내용_원문"]),
                "target_draft": _safe_str(row["지원대상_초벌요약"]),
                "content_draft": _safe_str(row["지원내용_초벌요약"]),
            }

            batch_row_ids.append(i)
            batch_inputs.append(input_data)
            rows_cache.append([i] + [row[c] for c in df.columns])
            i += 1

        next_start = i if i > next_start else next_start + 1
        if not batch_row_ids:
            continue

        summaries = generate_batch(batch_inputs)

        for k, base_vals in enumerate(rows_cache):
            out_row = base_vals + [summaries[k]]
            writer.writerow(out_row)
            processed_ids.add(base_vals[0])
            processed += 1

        f_csv.flush()
        print(f"... 진행 {processed}/{total} (최근 배치 row_id: {batch_row_ids[0]}~{batch_row_ids[-1]})")

print("[POST] CSV 쓰기 완료.")

# ─────────────────────────────────────────────────────────
# 9) 후처리: CSV → XLSX 변환
# ─────────────────────────────────────────────────────────
if not _stop:
    print("[POST] CSV → XLSX 변환 중(최종 산출물 생성)...")
    try:
        df_out = pd.read_csv(OUT_CSV_PATH, encoding="utf-8-sig")
        if ROW_ID_COL in df_out.columns:
            df_out = df_out.sort_values(by=[ROW_ID_COL]).reset_index(drop=True)
        df_out.to_excel(OUTPUT_XLSX_PATH, index=False)
        print(f"[DONE] 최종 XLSX 저장 완료: {OUTPUT_XLSX_PATH}")
    except Exception as e:
        print(f"[POST][WARN] XLSX 변환 실패(CSV는 저장됨): {e}")
else:
    print("[POST] 작업이 중단되었습니다. XLSX 변환을 건너뜁니다.")
    print(f"       (중단된 지점까지의 결과는 {OUTPUT_CSV_PATH} 에 저장됨)")

print(f"[END] 전체 완료 (경과: {time.time()-t0:.1f}s)")


Mounted at /content/drive
[GDRIVE] Mounted at /content/drive
[XLSX] Loaded: /content/drive/MyDrive/여성맞춤정책_요약_2차_결과_병합.xlsx (rows=1308)

[EDA] NA 비율 (주요 텍스트 컬럼):
지원대상_원문      0.002294
지원내용_원문      0.002294
지원대상_초벌요약    0.002294
지원내용_초벌요약    0.003058
dtype: float64

[EDA] 텍스트 길이 분포:

  ▷ 지원대상_원문 길이 통계:
count    1308.000000
mean      172.374618
std       254.239696
min         3.000000
25%        39.000000
50%        88.000000
75%       197.000000
max      2996.000000
Name: 지원대상_원문, dtype: float64

  ▷ 지원내용_원문 길이 통계:
count    1308.000000
mean      180.064985
std       230.973121
min         3.000000
25%        54.000000
50%       104.000000
75%       202.750000
max      1851.000000
Name: 지원내용_원문, dtype: float64

[EDA] 라벨 컬럼 'label' 이(가) 없어서 라벨 분포/학습 파트는 스킵됩니다.

[TRAIN] 학습 기능 비활성화이거나, 라벨 컬럼이 없어 학습을 건너뜁니다.
[WARN] LangGraph import 실패 → LangGraph 없이 진행합니다: module 'langgraph' has no attribute '__version__'

====== RUNTIME CHECK (요약용) ======
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

[QNT] 4-bit quantization enabled.
[LOAD] Primary model loaded.

[LG][WARN] LangGraph 없이 순차/배치 for-loop로 요약을 생성합니다.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[RESUME] (정상요약 기준) 처리 완료된 row_id 수: 956
[RUN] 요약 시작 (model=google/gemma-2-9b-it | batch=4)
... 진행 960/1308 (최근 배치 row_id: 956~959)
... 진행 964/1308 (최근 배치 row_id: 960~963)
... 진행 968/1308 (최근 배치 row_id: 964~967)
... 진행 972/1308 (최근 배치 row_id: 968~971)
... 진행 976/1308 (최근 배치 row_id: 972~975)
... 진행 980/1308 (최근 배치 row_id: 976~979)
... 진행 984/1308 (최근 배치 row_id: 980~983)
... 진행 988/1308 (최근 배치 row_id: 984~987)
... 진행 992/1308 (최근 배치 row_id: 988~991)
... 진행 996/1308 (최근 배치 row_id: 992~995)
... 진행 1000/1308 (최근 배치 row_id: 996~999)
... 진행 1004/1308 (최근 배치 row_id: 1000~1003)
... 진행 1008/1308 (최근 배치 row_id: 1004~1007)
... 진행 1012/1308 (최근 배치 row_id: 1008~1011)
... 진행 1016/1308 (최근 배치 row_id: 1012~1015)
... 진행 1020/1308 (최근 배치 row_id: 1016~1019)
... 진행 1024/1308 (최근 배치 row_id: 1020~1023)
... 진행 1028/1308 (최근 배치 row_id: 1024~1027)
... 진행 1032/1308 (최근 배치 row_id: 1028~1031)
... 진행 1036/1308 (최근 배치 row_id: 1032~1035)
... 진행 1040/1308 (최근 배치 row_id: 1036~1039)
... 진행 1044/1308 (최근 배치 row_id: 1040~1

In [ ]:
# 0) Java 설치
!pip install konlpy


In [ ]:
# -*- coding: utf-8 -*-
"""
policy_cls_train_and_eval_eda_no_trainer.py

- Colab + Google Drive 환경 전제
- 여성맞춤정책 엑셀/CSV를 자동으로 찾아서
  1) 분류 모델(klue/roberta-base) 학습 (체크포인트 없을 때만, 순수 PyTorch 루프)
  2) 체크포인트 저장 (/content/drive/MyDrive/policy_train_runs/checkpoints)
  3) 전체 데이터 기준 성능 평가 + EDA
     - Accuracy, Macro F1
     - ROC-AUC (macro/micro)
     - NDCG
     - classification_report
     - Confusion Matrix (ID-only + 상위 K 라벨 시각화)
     - ROC Curves (micro/macro + 일부 클래스)
"""

# ─────────────────────────────────────────────────────
# 0) Colab: 한글 폰트 설치 + Matplotlib 캐시 삭제
# ─────────────────────────────────────────────────────
!apt-get update -qq
!apt-get install -y fonts-nanum > /dev/null
!rm -rf ~/.cache/matplotlib  # 폰트 캐시 제거

from __future__ import annotations
import os
import sys
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    auc,
    ndcg_score,
)

# ─────────────────────────────────────────────────────
# Matplotlib + 폰트 설정 (NanumGothic 강제)
# ─────────────────────────────────────────────────────
import matplotlib as mpl
mpl.use("Agg")  # 화면 출력 없이 파일로만 저장 (Colab 서버용)
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

NANUM_GOTHIC_PATH = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

if os.path.exists(NANUM_GOTHIC_PATH):
    fm.fontManager.addfont(NANUM_GOTHIC_PATH)
    mpl.rcParams["font.family"] = "NanumGothic"
    mpl.rcParams["axes.unicode_minus"] = False
    print("[FONT] 사용 폰트 =", mpl.rcParams["font.family"])
else:
    print(f"[FONT][WARN] NanumGothic 폰트 파일을 찾지 못했습니다: {NANUM_GOTHIC_PATH}")
    print("[FONT][WARN] 기본 폰트로 진행합니다. 한글이 깨질 수 있습니다.")

# ─────────────────────────────────────────────────────
# 0-1) Colab 여부 + Drive 마운트
# ─────────────────────────────────────────────────────
IN_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()

if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive").exists():
            drive.mount("/content/drive")
            print("[GDRIVE] Mounted at /content/drive")
        else:
            print("[GDRIVE] Drive already mounted.")
    except Exception as e:
        print(f"[GDRIVE][WARN] Drive mount 실패(무시 가능): {e}")
else:
    print("[INFO] Not in Colab 환경. Drive 마운트는 생략합니다.")

# ─────────────────────────────────────────────────────
# 1) 기본 설정 (텍스트/라벨 컬럼 이름 등)
# ─────────────────────────────────────────────────────

# 텍스트/라벨 컬럼 이름 (실제 엑셀 기준)
TEXT_COL_TRAIN  = "지원내용_원문"
LABEL_COL_TRAIN = "카테고리_분류"

# 입력 파일 이름에 공통으로 들어갈 키워드 (엑셀/CSV 자동 탐색용)
INPUT_FILE_KEYWORD = "여성맞춤정책"

# 기본으로 먼저 시도해볼 경로 (없으면 자동 탐색)
DEFAULT_INPUT_FILE_PATH = "/content/drive/MyDrive/여성맞춤정책_요약_2차_결과_병합.xlsx"

# 학습 결과/체크포인트 루트
TRAIN_ROOT_DIR = Path("/content/drive/MyDrive/policy_train_runs")
CKPT_DIR       = TRAIN_ROOT_DIR / "checkpoints"

# 학습 메트릭 CSV (train/val/test 요약)
TRAIN_METRIC_CSV = TRAIN_ROOT_DIR / "metrics_cls.csv"

# 성능 EDA 결과 저장 위치
EVAL_OUT_DIR = TRAIN_ROOT_DIR / "eval_eda"

# 평가 시 배치 크기
EVAL_BATCH_SIZE = 32

# 학습 관련 하이퍼파라미터
NUM_EPOCHS  = 3
LEARNING_RATE = 5e-5

# 분류 모델 이름 (HF 허브)
CLS_MODEL_NAME = "klue/roberta-base"

# Confusion Matrix에서 한글 라벨까지 보여줄 상위 K 라벨 수
TOP_K_LABELS_FOR_CM = 20

# 디바이스 설정
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[DEVICE]", DEVICE)

# 시드 고정
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ─────────────────────────────────────────────────────
# 3) 엑셀/CSV 자동 탐색 + 로드
# ─────────────────────────────────────────────────────

def find_input_file(
    default_path: str,
    keyword: str,
    roots: List[Path],
) -> Path:
    """
    1순위: default_path 존재하면 그대로 사용
    2순위: roots 아래에서 keyword+(.xlsx|.xls|.csv) 패턴으로 검색
           → 문자열 길이가 가장 짧은 경로를 채택
    """
    default = Path(default_path)
    if default.is_file():
        print(f"[INPUT] 직접 지정한 파일 경로 사용: {default}")
        return default

    print(f"[INPUT][WARN] 기본 경로에서 파일을 찾지 못했습니다: {default}")
    print("[INPUT] 드라이브 전체에서 자동으로 엑셀/CSV 파일을 검색합니다...")

    candidates: List[Path] = []
    exts = (".xlsx", ".xls", ".csv")
    for root in roots:
        if not root.exists():
            continue
        for ext in exts:
            for p in root.rglob(f"*{keyword}*{ext}"):
                candidates.append(p)

    if not candidates:
        print("[INPUT][ERR] 입력 파일을 찾지 못했습니다.", file=sys.stderr)
        print("  - 검색 키워드:", keyword, file=sys.stderr)
        print("  - 검색 루트:", [str(r) for r in roots], file=sys.stderr)
        sys.exit(1)

    candidates = sorted(candidates, key=lambda x: len(str(x)))
    chosen = candidates[0]
    print("[INPUT][AUTO] 자동으로 선택된 파일:")
    for i, p in enumerate(candidates[:5]):
        mark = "→ 사용" if i == 0 else "   참고"
        print(f"  {mark}: {p}")
    return chosen

search_roots = [Path("/content/drive/MyDrive"), Path("/content/drive")]
INPUT_PATH = find_input_file(DEFAULT_INPUT_FILE_PATH, INPUT_FILE_KEYWORD, search_roots)

if INPUT_PATH.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(INPUT_PATH)
elif INPUT_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
else:
    print(f"[INPUT][ERR] 지원하지 않는 파일 확장자입니다: {INPUT_PATH.suffix}", file=sys.stderr)
    sys.exit(1)

print(f"[XLSX/CSV] Loaded: {INPUT_PATH} (rows={len(df)})")

if TEXT_COL_TRAIN not in df.columns or LABEL_COL_TRAIN not in df.columns:
    print(f"[ERR] 필요한 컬럼이 없습니다: {TEXT_COL_TRAIN}, {LABEL_COL_TRAIN}", file=sys.stderr)
    print("[INFO] 현재 컬럼:", df.columns.tolist(), file=sys.stderr)
    sys.exit(1)

df = df.dropna(subset=[TEXT_COL_TRAIN, LABEL_COL_TRAIN]).reset_index(drop=True)
print(f"[DATA] NA 제거 후 shape: {df.shape}")

print("\n[EDA] 라벨 분포:")
print(df[LABEL_COL_TRAIN].value_counts())

# 라벨 인코딩
le = LabelEncoder()
df["label_id"] = le.fit_transform(df[LABEL_COL_TRAIN].astype(str))
num_labels = len(le.classes_)
print("\n[INFO] 라벨 인코딩 결과 (id → label):")
for i, label_name in enumerate(le.classes_):
    print(f"  {i} → {label_name}")

# ─────────────────────────────────────────────────────
# 4) Dataset 정의
# ─────────────────────────────────────────────────────
class TextClsDataset(Dataset):
    def __init__(self, df: pd.DataFrame, text_col: str, label_col: str):
        self.df = df.reset_index(drop=True)
        self.text_col = text_col
        self.label_col = label_col

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        return {
            "text": str(row[self.text_col]),
            "labels": int(row[self.label_col]),
        }

# ─────────────────────────────────────────────────────
# 5) 토크나이저/모델 로드 + 수동 학습 루프
# ─────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CKPT_DIR.mkdir(parents=True, exist_ok=True)

def build_tokenizer_and_model() -> Tuple[AutoTokenizer, AutoModelForSequenceClassification]:
    print(f"[MODEL] base model: {CLS_MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(CLS_MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        CLS_MODEL_NAME,
        num_labels=num_labels,
    )
    model.to(DEVICE)
    return tokenizer, model

def compute_metrics_np(logits: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="macro")

    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

    try:
        y_true_onehot = np.eye(num_labels)[labels]
        roc_auc_macro = roc_auc_score(y_true_onehot, probs, multi_class="ovr", average="macro")
    except Exception as e:
        print(f"[METRIC][WARN] ROC-AUC 계산 실패: {e}")
        roc_auc_macro = float("nan")

    try:
        y_true_onehot = np.eye(num_labels)[labels]
        ndcg_val = ndcg_score(y_true_onehot, probs)
    except Exception as e:
        print(f"[METRIC][WARN] NDCG 계산 실패: {e}")
        ndcg_val = float("nan")

    return {
        "accuracy": acc,
        "f1_macro": f1,
        "roc_auc_macro": roc_auc_macro,
        "ndcg": ndcg_val,
    }

def make_loader(df_part: pd.DataFrame, tokenizer, batch_size: int, shuffle: bool) -> DataLoader:
    ds = TextClsDataset(df_part, TEXT_COL_TRAIN, "label_id")

    def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        texts = [b["text"] for b in batch]
        labels = torch.tensor([b["labels"] for b in batch], dtype=torch.long)
        enc = tokenizer(
            texts,
            truncation=True,
            max_length=256,
            padding=True,
            return_tensors="pt",
        )
        enc["labels"] = labels
        return enc

    loader = DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)
    return loader

def evaluate_model(model, loader) -> Tuple[np.ndarray, np.ndarray, float]:
    model.eval()
    all_logits: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []
    total_loss = 0.0
    steps = 0

    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels").to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch, labels=labels)
            loss = outputs.loss
            logits = outputs.logits.detach().cpu().numpy()
            all_logits.append(logits)
            all_labels.append(labels.detach().cpu().numpy())
            total_loss += float(loss.item())
            steps += 1

    logits_np = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0)
    avg_loss = total_loss / max(steps, 1)
    return logits_np, labels_np, avg_loss

def train_if_needed_and_load():
    # 이미 학습된 모델이 있는지 판단
    ckpt_config = CKPT_DIR / "config.json"
    ckpt_model  = CKPT_DIR / "pytorch_model.bin"

    if ckpt_config.is_file() and ckpt_model.is_file():
        print(f"[CKPT] 기존 체크포인트 발견 → 재학습 없이 로드합니다: {CKPT_DIR}")
        tokenizer = AutoTokenizer.from_pretrained(str(CKPT_DIR))
        model = AutoModelForSequenceClassification.from_pretrained(str(CKPT_DIR))
        model.to(DEVICE)
        return tokenizer, model

    print("[CKPT] 체크포인트가 없습니다 → 새로 학습을 시작합니다.")

    tokenizer, model = build_tokenizer_and_model()

    from sklearn.model_selection import train_test_split
    try:
        tr_df, tmp_df = train_test_split(
            df,
            test_size=0.2,
            stratify=df["label_id"],
            random_state=42,
        )
        val_df, te_df = train_test_split(
            tmp_df,
            test_size=0.5,
            stratify=tmp_df["label_id"],
            random_state=42,
        )
    except ValueError as e:
        print(f"[SPLIT][WARN] stratify 실패 → 일반 분할 사용: {e}")
        tr_df, tmp_df = train_test_split(
            df,
            test_size=0.2,
            random_state=42,
        )
        val_df, te_df = train_test_split(
            tmp_df,
            test_size=0.5,
            random_state=42,
        )

    print(f"[SPLIT] train={len(tr_df)}, val={len(val_df)}, test={len(te_df)}")

    train_loader = make_loader(tr_df, tokenizer, batch_size=16, shuffle=True)
    val_loader   = make_loader(val_df, tokenizer, batch_size=32, shuffle=False)
    test_loader  = make_loader(te_df, tokenizer, batch_size=32, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

    best_val_loss = float("inf")
    best_state_dict = None

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n[TRAIN] Epoch {epoch}/{NUM_EPOCHS}")
        model.train()
        total_loss = 0.0
        steps = 0

        for batch in train_loader:
            labels = batch.pop("labels").to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(**batch, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_loss += float(loss.item())
            steps += 1

            if steps % 50 == 0:
                print(f"  - step {steps}, loss={total_loss/steps:.4f}")

        avg_train_loss = total_loss / max(steps, 1)
        print(f"[TRAIN] Epoch {epoch} 평균 train loss = {avg_train_loss:.4f}")

        # validation
        val_logits, val_labels, val_loss = evaluate_model(model, val_loader)
        val_metrics = compute_metrics_np(val_logits, val_labels)
        print(f"[VAL]  loss={val_loss:.4f}, "
              f"acc={val_metrics['accuracy']:.4f}, "
              f"f1={val_metrics['f1_macro']:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print("[VAL] 현재까지 최고의 모델로 갱신")

    # best 모델로 되돌리기
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        print("[CKPT] best validation loss 기준 모델 로드 완료")

    # 최종적으로 train/val/test 전체 메트릭 계산해서 저장
    train_logits, train_labels, train_loss = evaluate_model(model, train_loader)
    val_logits, val_labels, val_loss = evaluate_model(model, val_loader)
    test_logits, test_labels, test_loss = evaluate_model(model, test_loader)

    train_metrics = compute_metrics_np(train_logits, train_labels)
    val_metrics   = compute_metrics_np(val_logits, val_labels)
    test_metrics  = compute_metrics_np(test_logits, test_labels)

    metrics_df = pd.DataFrame([
        {"split": "train", "loss": train_loss, **train_metrics},
        {"split": "val",   "loss": val_loss,   **val_metrics},
        {"split": "test",  "loss": test_loss,  **test_metrics},
    ])
    TRAIN_ROOT_DIR.mkdir(parents=True, exist_ok=True)
    metrics_df.to_csv(TRAIN_METRIC_CSV, index=False, encoding="utf-8-sig")
    print(f"[TRAIN] metrics_cls.csv 저장: {TRAIN_METRIC_CSV}")

    # 체크포인트 저장
    model.save_pretrained(str(CKPT_DIR))
    tokenizer.save_pretrained(str(CKPT_DIR))
    print(f"[CKPT] 모델/토크나이저 저장 완료: {CKPT_DIR}")

    return tokenizer, model

tokenizer, model = train_if_needed_and_load()
model.to(DEVICE)
model.eval()

# ─────────────────────────────────────────────────────
# 6) 전체 데이터에 대한 예측/로짓 수집 (Full EVAL)
# ─────────────────────────────────────────────────────
full_loader = make_loader(df, tokenizer, batch_size=EVAL_BATCH_SIZE, shuffle=False)

print("\n[EVAL] 전체 데이터에 대해 예측 중...")
logits, labels, eval_loss = evaluate_model(model, full_loader)
print(f"[EVAL] full-data 평균 loss = {eval_loss:.4f}")
print(f"[EVAL] logits shape = {logits.shape}, labels shape = {labels.shape}")

probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
preds = np.argmax(logits, axis=-1)

# ─────────────────────────────────────────────────────
# 7) 기본 지표 계산 (Accuracy, F1, ROC-AUC, NDCG)
# ─────────────────────────────────────────────────────
acc = accuracy_score(labels, preds)
f1_macro = f1_score(labels, preds, average="macro")

y_true_onehot = np.eye(num_labels)[labels]
try:
    roc_auc_macro = roc_auc_score(y_true_onehot, probs, multi_class="ovr", average="macro")
    roc_auc_micro = roc_auc_score(y_true_onehot, probs, multi_class="ovr", average="micro")
except Exception as e:
    print(f"[WARN] ROC-AUC 계산 실패: {e}")
    roc_auc_macro = float("nan")
    roc_auc_micro = float("nan")

try:
    ndcg = ndcg_score(y_true_onehot, probs)
except Exception as e:
    print(f"[WARN] NDCG 계산 실패: {e}")
    ndcg = float("nan")

print("\n===== [SUMMARY METRICS - FULL DATA] =====")
print(f"Accuracy          : {acc:.4f}")
print(f"Macro F1          : {f1_macro:.4f}")
print(f"ROC-AUC (macro)   : {roc_auc_macro:.4f}")
print(f"ROC-AUC (micro)   : {roc_auc_micro:.4f}")
print(f"NDCG (full)       : {ndcg:.4f}")
print("=========================================\n")

# ─────────────────────────────────────────────────────
# 8) 라벨별 지표 + 혼동행렬 + ROC Curve (한글 폰트 적용)
# ─────────────────────────────────────────────────────
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# 8-1) classification_report
report_dict = classification_report(
    labels,
    preds,
    target_names=list(le.classes_),
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T
report_csv_path = EVAL_OUT_DIR / "classification_report.csv"
report_df.to_csv(report_csv_path, encoding="utf-8-sig")
print(f"[SAVE] classification_report.csv 저장: {report_csv_path}")

# 8-2) confusion matrix (CSV + ID 기반 그림 + 상위 K 라벨 그림)
cm = confusion_matrix(labels, preds)

# (1) 전체 혼동행렬 CSV 저장
cm_csv_path = EVAL_OUT_DIR / "confusion_matrix.csv"
np.savetxt(cm_csv_path, cm, fmt="%d", delimiter=",")
print(f"[SAVE] confusion_matrix.csv 저장: {cm_csv_path}")

# (2) label_id ↔ 한글 라벨 매핑 CSV 저장
label_map_df = pd.DataFrame({
    "label_id": np.arange(num_labels),
    "label_name": le.classes_,
})
label_map_path = EVAL_OUT_DIR / "label_id_mapping.csv"
label_map_df.to_csv(label_map_path, index=False, encoding="utf-8-sig")
print(f"[SAVE] label_id_mapping.csv 저장: {label_map_path}")

# (3) 전체 라벨을 ID만으로 표시한 혼동행렬
plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix (ID only)")
plt.colorbar()

tick_marks = np.arange(num_labels)
plt.xticks(tick_marks, tick_marks, rotation=90, fontsize=6)
plt.yticks(tick_marks, tick_marks, fontsize=6)
plt.xlabel("Predicted label_id")
plt.ylabel("True label_id")
plt.tight_layout()

cm_idx_png_path = EVAL_OUT_DIR / "confusion_matrix_id_only.png"
plt.savefig(cm_idx_png_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"[SAVE] confusion_matrix_id_only.png 저장: {cm_idx_png_path}")

# (4) 출현 수가 많은 TOP-K 라벨만 한글 라벨까지 붙여서 보기 좋게
from collections import Counter
label_freq = Counter(labels)
top_ids = [lid for lid, _ in label_freq.most_common(TOP_K_LABELS_FOR_CM)]
top_ids = np.array(sorted(top_ids))

cm_top = cm[np.ix_(top_ids, top_ids)]
top_names = [f"{i}: {le.classes_[i]}" for i in top_ids]

plt.figure(figsize=(max(8, 0.4 * len(top_ids)), max(6, 0.4 * len(top_ids))))
plt.imshow(cm_top, interpolation="nearest")
plt.title(f"Confusion Matrix (Top {len(top_ids)} labels)")
plt.colorbar()

ticks = np.arange(len(top_ids))
plt.xticks(ticks, top_names, rotation=90, fontsize=6)
plt.yticks(ticks, top_names, fontsize=6)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()

cm_top_png_path = EVAL_OUT_DIR / f"confusion_matrix_top{len(top_ids)}.png"
plt.savefig(cm_top_png_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"[SAVE] confusion_matrix_top{len(top_ids)}.png 저장: {cm_top_png_path}")

# 8-3) ROC Curves
try:
    fpr_micro, tpr_micro, _ = roc_curve(y_true_onehot.ravel(), probs.ravel())
    roc_auc_micro_curve = auc(fpr_micro, tpr_micro)

    fpr_dict = {}
    tpr_dict = {}
    roc_auc_dict = {}
    for i in range(num_labels):
        fpr_dict[i], tpr_dict[i], _ = roc_curve(y_true_onehot[:, i], probs[:, i])
        roc_auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

    all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in range(num_labels)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(num_labels):
        mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
    mean_tpr /= num_labels
    roc_auc_macro_curve = auc(all_fpr, mean_tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(
        fpr_micro,
        tpr_micro,
        linestyle="--",
        label=f"micro-average ROC (AUC = {roc_auc_micro_curve:.3f})",
    )
    plt.plot(
        all_fpr,
        mean_tpr,
        label=f"macro-average ROC (AUC = {roc_auc_macro_curve:.3f})",
    )

    max_per_class_plot = min(5, num_labels)
    for i in range(max_per_class_plot):
        plt.plot(
            fpr_dict[i],
            tpr_dict[i],
            alpha=0.7,
            label=f"Class {i} ({le.classes_[i]}) (AUC = {roc_auc_dict[i]:.3f})",
        )

    plt.plot([0, 1], [0, 1], "k--", label="Random")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves (micro/macro + 일부 클래스)")
    plt.legend(fontsize=8, loc="lower right")
    plt.tight_layout()

    roc_png_path = EVAL_OUT_DIR / "roc_curves.png"
    plt.savefig(roc_png_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"[SAVE] roc_curves.png 저장: {roc_png_path}")
except Exception as e:
    print(f"[WARN] ROC Curve 그리기 실패 (무시 가능): {e}")

# ─────────────────────────────────────────────────────
# 9) train/val/test 메트릭 CSV 있으면 같이 보여주기
# ─────────────────────────────────────────────────────
if TRAIN_METRIC_CSV.is_file():
    print(f"\n[INFO] 기존 학습 메트릭 CSV: {TRAIN_METRIC_CSV}")
    metric_df = pd.read_csv(TRAIN_METRIC_CSV)
    print(metric_df)
else:
    print(f"\n[INFO] {TRAIN_METRIC_CSV} 를 찾지 못했습니다. (학습 메트릭 CSV 없음)")

# ─────────────────────────────────────────────────────
# 10) 전체 평가 요약 CSV 저장
# ─────────────────────────────────────────────────────
summary_row = {
    "accuracy": acc,
    "f1_macro": f1_macro,
    "roc_auc_macro": roc_auc_macro,
    "roc_auc_micro": roc_auc_micro,
    "ndcg": ndcg,
}

summary_df = pd.DataFrame([summary_row])
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
summary_csv_path = EVAL_OUT_DIR / "eval_summary.csv"
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")
print(f"\n[SAVE] eval_summary.csv 저장: {summary_csv_path}")

print("\n[END] 학습(필요 시) + 성능 평가 + EDA 전체 완료.")


In [ ]:
# -*- coding: utf-8 -*-
"""
여성맞춤정책_요약 결과 EDA + 시각화 자동화
- Colab 전용
- 한글 폰트(NanumGothic) 설정
- Java 자동 탐색 + konlpy Okt 사용
- 입력 데이터: policy_summary_langchain_streaming.csv
"""

# 0️⃣ 시스템 세팅: 폰트 + Java 설치 (Colab 기준)
!apt-get update -qq
# openjdk-17-jdk-headless를 설치하여 JVM 환경을 설정합니다.
# fonts-nanum은 그래프에서 한글이 깨지지 않도록 하기 위함입니다.
!apt-get install -y fonts-nanum openjdk-17-jdk-headless > /dev/null
!rm -rf ~/.cache/matplotlib  # matplotlib 폰트 캐시를 지워 폰트 변경 사항을 적용합니다.

# ─────────────────────────────────────────────────────────
# Python package installation for konlpy and JPype1
# 기존 설치된 konlpy와 JPype1이 있을 경우 충돌을 방지하기 위해 먼저 제거합니다.
!pip uninstall -y konlpy JPype1
!pip install konlpy JPype1

# ─────────────────────────────────────────────────────────
# 1️⃣ JAVA_HOME 및 libjvm.so 경로 자동 설정 (JVM 경로 자동 찾기)
import os
import glob
from pathlib import Path
import jpype

# JVM 관련 경로 변수 초기화
found_java_home_dir = None
found_jvm_lib_path = None
_konlpy_ready = False

# 일반적인 JDK 설치 경로들을 탐색 (최신 버전 우선)
jvm_dirs = sorted(glob.glob("/usr/lib/jvm/java-*-openjdk-amd64"), reverse=True)

for candidate_dir_str in jvm_dirs:
    candidate_dir = Path(candidate_dir_str)
    # libjvm.so 파일의 일반적인 위치들을 확인합니다.
    libjvm_search_paths = [
        candidate_dir / "jre" / "lib" / "amd64" / "server" / "libjvm.so",
        candidate_dir / "lib" / "server" / "libjvm.so",
        candidate_dir / "lib" / "amd64" / "server" / "libjvm.so",
        candidate_dir / "lib" / "jli" / "libjvm.so"  # 다른 가능한 위치
    ]
    for lib_path in libjvm_search_paths:
        if lib_path.exists():
            found_java_home_dir = str(candidate_dir)
            found_jvm_lib_path = str(lib_path)
            break
    if found_jvm_lib_path:
        break

# libjvm.so 경로가 성공적으로 찾아졌을 경우 환경 변수를 설정합니다.
if found_jvm_lib_path:
    os.environ["JAVA_HOME"] = found_java_home_dir
    os.environ["PATH"] += os.pathsep + os.path.join(found_java_home_dir, "bin")
    print(f"[JAVA] JAVA_HOME 설정: {os.environ['JAVA_HOME']}")
    print(f"[JAVA] libjvm.so 경로: {found_jvm_lib_path}")
    _konlpy_ready = True
else:
    # JPype의 기본 JVM 경로를 시도 (덜 안정적일 수 있음)
    try:
        default_jvm_path = jpype.getDefaultJVMPath()
        if default_jvm_path:
            found_jvm_lib_path = default_jvm_path
            # JAVA_HOME을 libjvm.so 경로로부터 추정하여 설정합니다.
            # (일반적으로 libjvm.so는 JAVA_HOME/jre/lib/amd64/server 에 위치하므로 3단계 상위 디렉토리)
            inferred_java_home = str(Path(found_jvm_lib_path).parents[3])
            os.environ["JAVA_HOME"] = inferred_java_home
            print(f"[JAVA] libjvm.so 경로 (JPype 기본값): {found_jvm_lib_path}")
            print(f"[JAVA] JAVA_HOME (추정): {os.environ['JAVA_HOME']}")
            _konlpy_ready = True
        else:
            print("[JAVA] WARN: JPype의 기본 JVM 경로도 찾을 수 없습니다.")
            _konlpy_ready = False
    except jpype.JVMNotFoundException:
        print("[JAVA] WARN: JPype의 기본 JVM 경로도 찾을 수 없습니다.")
        _konlpy_ready = False

if not _konlpy_ready:
    print("[JAVA] ERROR: JVM shared library (libjvm.so) 파일을 찾을 수 없어 konlpy 초기화 실패.")

# ─────────────────────────────────────────────────────
# 0-1) Colab 여부 + Drive 마운트
# ─────────────────────────────────────────────────────
IN_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()

if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive").exists():
            drive.mount("/content/drive")
            print("[GDRIVE] Mounted at /content/drive")
        else:
            print("[GDRIVE] Drive already mounted.")
    except Exception as e:
        print(f"[GDRIVE][WARN] Drive mount 실패(무시 가능): {e}")
else:
    print("[INFO] Not in Colab 환경. Drive 마운트는 생략합니다.")

# 2️⃣ 폰트 설정 (NanumGothic 사용)
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

NANUM_GOTHIC_PATH = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(NANUM_GOTHIC_PATH):
    print(f"[FONT][WARN] 폰트 파일을 찾을 수 없습니다: {NANUM_GOTHIC_PATH}. 시스템 폰트를 재검색합니다.")
    fm._load_fontmanager(try_read_cache=False)  # 폰트 캐시를 지우고 다시 로드합니다.
    if os.path.exists(NANUM_GOTHIC_PATH):
        print("[FONT] 폰트 재검색 후 발견.")
    else:
        print("[FONT][ERROR] NanumGothic 폰트를 최종적으로 찾을 수 없습니다. 그래프에 한글이 깨질 수 있습니다.")
        NANUM_GOTHIC_PATH = mpl.rcParams['font.sans-serif'][0]  # 기본 폰트로 폴백

if os.path.exists(NANUM_GOTHIC_PATH):  # 재확인 후 폰트 설정
    fm.fontManager.addfont(NANUM_GOTHIC_PATH)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지
    print("[FONT] 사용 폰트 =", plt.rcParams["font.family"])
else:
    print("[FONT] 경고: 한글 폰트 설정에 실패했습니다. 그래프에 한글이 깨질 수 있습니다.")

# ───────────────────────────────────────────────
# 3️⃣ EDA + 시각화 코드
# ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# konlpy + Okt 초기화 (Java 설정이 완료되었을 경우)
from konlpy.tag import Okt

if _konlpy_ready:
    # 명시적으로 JVM 경로를 Okt 생성자에 전달합니다.
    okt = Okt(jvmpath=found_jvm_lib_path)
    print("[KONLPY] Okt 로드 완료")
else:
    print("[KONLPY] WARN: JVM 로드 실패로 Okt를 초기화할 수 없습니다.")
    okt = None  # 오류 방지를 위해 okt를 None으로 설정합니다.

# 4️⃣ 입력 데이터(CSV) 경로
INPUT_PATH = Path("/content/drive/MyDrive/policy_summary_langchain_streaming.csv")

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {INPUT_PATH}")

df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
print(f"[LOAD] {INPUT_PATH.name} 불러오기 완료 (rows={len(df)}, cols={len(df.columns)})")

# 주요 텍스트 컬럼 (원문/요약 관련)
text_cols = [c for c in df.columns if any(k in c for k in ["원문", "요약"])]
print(f"[INFO] 텍스트 관련 컬럼: {text_cols}")

# 결측치 / 중복
print("\n[EDA] 결측치 비율:")
print(df[text_cols].isna().mean().sort_values(ascending=False))

print("\n[EDA] 중복 행 수:")
print(df.duplicated(subset=text_cols).sum())

# 텍스트 길이 분석
for col in text_cols:
    df[f"{col}_len"] = df[col].astype(str).apply(len)

len_cols = [c for c in df.columns if c.endswith("_len")]

# 길이 컬럼들을 숫자로 강제 변환 (object 타입 방지)
for col in len_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\n[EDA] 텍스트 길이 통계:")
print(df[len_cols].describe())

# 5️⃣ 텍스트 길이 분포 그래프
plt.figure(figsize=(10, 6))
for col in len_cols:
    sns.kdeplot(df[col].dropna(), label=col, fill=True, alpha=0.3)
plt.title("텍스트 길이 분포 비교")
plt.xlabel("문자 수")
plt.ylabel("밀도")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 6️⃣ 길이 상관관계 Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df[len_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("텍스트 길이 상관관계 Heatmap")
plt.show()

# 7️⃣ 워드클라우드 (최종요약 컬럼 기준)
summary_col_candidates = [c for c in df.columns if "최종요약" in c]
summary_col = None

if summary_col_candidates and okt:  # okt가 성공적으로 초기화되었는지 확인
    summary_col = summary_col_candidates[0]
    print(f"\n[WORDCLOUD] 사용 요약 컬럼: {summary_col}")

    all_text = " ".join(df[summary_col].dropna().astype(str))

    # 1차: Okt로 명사만 추출 (길이 2자 이상)
    nouns = [n for n in okt.nouns(all_text) if len(n) > 1]
    freq = Counter(nouns)

    # 만약 형태소 분석으로는 단어가 하나도 안 잡힌 경우 → 공백 기준 토큰으로 재시도
    if not freq:
        print("\n[WORDCLOUD] 형태소 분석으로 추출된 단어가 없어, 공백 기준 토큰으로 재시도합니다.")
        tokens = [t for t in all_text.split() if len(t) > 1]
        freq = Counter(tokens)

    # 그래도 단어가 없으면 워드클라우드 스킵
    if not freq:
        print("[WORDCLOUD] 사용할 단어가 없어 워드클라우드를 건너뜁니다.")
    else:
        print("\n[WORDCLOUD] 상위 단어:")
        print(pd.DataFrame(freq.most_common(20), columns=["단어", "빈도"]))

        wc = WordCloud(
            font_path=NANUM_GOTHIC_PATH,  # 실제 TTF 경로
            background_color="white",
            width=800,
            height=600,
            colormap="tab10"
        ).generate_from_frequencies(freq)

        plt.figure(figsize=(10, 7))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"요약문 워드클라우드 ({summary_col})")
        plt.show()
else:
    print("\n[WORDCLOUD] '최종요약' 이 들어간 컬럼을 찾지 못했거나, konlpy 초기화에 실패하여 워드클라우드를 생성할 수 없습니다.")

# 8️⃣ 텍스트 길이 간 단순 상관관계 시각화
pairplot_cols = [c for c in df.columns if c.endswith("_len")]
if len(pairplot_cols) >= 2:
    sns.pairplot(df[pairplot_cols], diag_kind="kde", corner=True)
    plt.suptitle("텍스트 길이 간 관계 Pairplot", y=1.02)
    plt.show()

# 9️⃣ 추가: 요약문 품질 기초 통계
# summary_col이 이전에 정의되지 않았을 경우를 대비
if summary_col is None:
    summary_col_candidates = [c for c in df.columns if "최종요약" in c]
    if summary_col_candidates:
        summary_col = summary_col_candidates[0]

if summary_col:
    len_col = f"{summary_col}_len"
    if len_col in df.columns:
        length_series = pd.to_numeric(df[len_col], errors="coerce")
        valid_mask = length_series.notna()

        if valid_mask.sum() == 0:
            print("[SUMMARY STATS] 유효한 요약문 길이 데이터가 없어 통계를 계산할 수 없습니다.")
        else:
            avg_len = length_series.mean()
            std_len = length_series.std()
            print(f"[SUMMARY STATS] 요약문 평균 길이: {avg_len:.1f} ± {std_len:.1f}")

            # 가장 긴/짧은 요약문 예시 추출
            long_idx = length_series.nlargest(3).index
            short_idx = length_series.nsmallest(3).index

            print("\n[요약문 예시: 가장 짧은 3개]")
            for i, idx in enumerate(short_idx, start=1):
                print(f"{i}. {df.loc[idx, summary_col]}\n")

            print("[요약문 예시: 가장 긴 3개]")
            for i, idx in enumerate(long_idx, start=1):
                print(f"{i}. {df.loc[idx, summary_col]}\n")
    else:
        print(f"[SUMMARY STATS] 길이 컬럼({len_col})을 찾을 수 없습니다.")
else:
    print("[SUMMARY STATS] '최종요약' 컬럼을 찾을 수 없어 통계를 계산하지 않습니다.")

print("\n[END] EDA + 시각화 완료 ✅")
